In [ ]:
import subprocess
import sys

packages = [
    'mediapipe',
    'opencv-python-headless',
    'ultralytics',
    'tqdm',
    'pandas',
    'numpy',
    'matplotlib',
    'ipywidgets',
    'kagglehub',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Dependencies installed successfully.')

In [3]:
# IaMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
enider_yawdd_dataset_path = kagglehub.dataset_download('enider/yawdd-dataset')

print('Data source import complete.')

c:\Users\Omswaroop\OneDrive\Desktop\New folder\.venv-1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 5.08G/5.08G [12:37<00:00, 7.21MB/s]  

Extracting files...


Data source import complete.


# YawDD -> 3-Class Frame Labeling Pipeline (Talking / Yawning / Mouth_Closed)
### Kaggle Notebook version

This notebook extracts frames from YawDD videos and labels each frame into one of:
- `mouth_closed`
- `talking`
- `yawning`

**Method**: face+mouth landmark extraction (MediaPipe Face Mesh) -> mouth-aspect-ratio (MAR) time
series per video -> temporal-duration rule to separate sustained wide-mouth (yawn) from brief
mouth-opening (talk) -> low-confidence frames flagged for human review.

**Important accuracy note (read before trusting the output):**
This pipeline reproduces the *method* used by the YawDD+ paper (Mujtaba et al., ICIP 2026), but
YawDD+'s own published annotations are binary (open/close mouth, i.e. yawn vs no-yawn) -- they do
**not** publish a 3-way talking/mouth_closed split. This notebook's talking vs mouth_closed
separation is a heuristic (based on mouth-opening variance/duration) that **must be calibrated and
spot-checked by a human** on your own data before you trust it for anything safety-critical. The
notebook includes a manual-review step for exactly this reason -- do not skip it.

## Kaggle setup notes
1. Add the YawDD dataset as a Kaggle **Input** (Add Data, top right) so it appears under
   `/kaggle/input/<dataset-name>/...`. Edit `VIDEO_ROOT` below to match.
2. Turn on **Internet** in notebook settings (needed for `pip install`).
3. If you want a GPU for speed, enable an accelerator in settings -- MediaPipe here runs on CPU
   either way, but it won't hurt other steps if you extend this later (e.g. CNN training).
4. All outputs are written under `/kaggle/working/` so they show up in the notebook's Output tab
   and can be saved as a Kaggle Dataset/versioned output when you commit the notebook.

## 1. Setup -- install dependencies

In [8]:
!pip install -q mediapipe opencv-python-headless ultralytics tqdm pandas numpy matplotlib ipywidgets

## 2. Point at your input data

Kaggle mounts input datasets read-only under `/kaggle/input/`. List what's available, then set
`VIDEO_ROOT` to the folder that actually contains the `.avi` files (adjust the dataset slug to
whatever you named it when adding the data).

In [1]:
import os

# Local fallback: use a folder named 'videos' in the notebook directory if present
NOTEBOOK_DIR = os.getcwd()
VIDEO_ROOT = os.path.join(NOTEBOOK_DIR, 'videos')

# If a Kaggle-style dataset path exists, use that instead
if not os.path.exists(VIDEO_ROOT):
    try:
        import kagglehub
        dataset_path = kagglehub.dataset_download('enider/yawdd-dataset')
        if os.path.exists(dataset_path):
            VIDEO_ROOT = dataset_path
    except Exception as e:
        print(f'Kaggle dataset download unavailable: {e}')

OUTPUT_ROOT = os.path.join(NOTEBOOK_DIR, 'frames')
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print(f'Video root set to: {VIDEO_ROOT}')

c:\Users\Omswaroop\OneDrive\Desktop\New folder\.venv-1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Video root set to: C:\Users\Omswaroop\.cache\kagglehub\datasets\enider\yawdd-dataset\versions\4


## 3. Find all videos

Recursively scans `VIDEO_ROOT` for `.avi` files and parses whatever state info is available from
the filename (helps sanity-check the automatic labels later, but is **not** used to directly
assign labels -- the whole point of frame-level labeling is to not trust filenames blindly).

In [2]:
import re, glob

def find_videos(root):
    return sorted(glob.glob(os.path.join(root, '**', '*.avi'), recursive=True))

def parse_filename_state(path):
    """Best-effort parse of YawDD filename convention: {ID}-{Gender}{Glasses}-{State}.avi
    Returns None if the filename doesn't carry a state (front-camera videos)."""
    name = os.path.splitext(os.path.basename(path))[0]
    m = re.match(r'^\d+-[A-Za-z]+-(.+)$', name)
    if not m:
        return None
    state = m.group(1).lower()
    if 'yawning' in state and 'talking' in state:
        return 'talkingyawning'
    if 'yawning' in state:
        return 'yawning'
    if 'talking' in state:
        return 'talking'
    if 'normal' in state:
        return 'normal'
    return None

videos = find_videos(VIDEO_ROOT)
print(f'Found {len(videos)} videos')
for v in videos[:10]:
    print(' -', v, '| filename state:', parse_filename_state(v))

assert len(videos) > 0, 'No .avi files found -- check VIDEO_ROOT matches your Kaggle input path'

Found 348 videos
 - C:\Users\Omswaroop\.cache\kagglehub\datasets\enider\yawdd-dataset\versions\4\Dash\Dash\Female\1-FemaleNoGlasses.avi | filename state: None
 - C:\Users\Omswaroop\.cache\kagglehub\datasets\enider\yawdd-dataset\versions\4\Dash\Dash\Female\10-FemaleNoGlasses.avi | filename state: None
 - C:\Users\Omswaroop\.cache\kagglehub\datasets\enider\yawdd-dataset\versions\4\Dash\Dash\Female\11-FemaleGlasses.avi.avi | filename state: None
 - C:\Users\Omswaroop\.cache\kagglehub\datasets\enider\yawdd-dataset\versions\4\Dash\Dash\Female\12-FemaleGlasses.avi.avi | filename state: None
 - C:\Users\Omswaroop\.cache\kagglehub\datasets\enider\yawdd-dataset\versions\4\Dash\Dash\Female\13-FemaleGlasses.avi.avi | filename state: None
 - C:\Users\Omswaroop\.cache\kagglehub\datasets\enider\yawdd-dataset\versions\4\Dash\Dash\Female\2-FemaleNoGlasses.avi | filename state: None
 - C:\Users\Omswaroop\.cache\kagglehub\datasets\enider\yawdd-dataset\versions\4\Dash\Dash\Female\3-FemaleGlasses.avi | fi

## 4. Face detection + mouth landmark extraction

Uses MediaPipe Face Mesh to locate the mouth region per frame. If multiple faces are detected
(passenger, background person), we pick the **largest bounding box**, matching the paper's
heuristic that the driver's face occupies the most area in dashboard/rear-view cameras.

In [3]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import urllib.request

# 1. Download the Face Landmarker model file
model_url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
model_path = "face_landmarker.task"
urllib.request.urlretrieve(model_url, model_path)

# Landmark indices for MAR calculation
UPPER_LIP = [61, 185, 40, 39, 37, 267, 269, 270, 409, 291]
LOWER_LIP = [0, 17, 314, 405, 321, 375, 89, 87, 14, 317, 402, 318]
MOUTH_ALL = UPPER_LIP + LOWER_LIP

def get_mouth_mar_and_crop(frame, detector):
    """Returns (mar, mouth_crop) using MediaPipe Tasks API."""
    h, w = frame.shape[:2]
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    detection_result = detector.detect(mp_image)

    if not detection_result.face_landmarks:
        return None, None

    # Pick the largest face (closest to driver heuristic)
    face = detection_result.face_landmarks[0]
    if len(detection_result.face_landmarks) > 1:
        # Simple area heuristic based on bounds
        def get_area(landmarks):
            xs = [p.x for p in landmarks]
            ys = [p.y for p in landmarks]
            return (max(xs) - min(xs)) * (max(ys) - min(ys))
        face = max(detection_result.face_landmarks, key=get_area)

    def pt(i):
        p = face[i]
        return np.array([p.x * w, p.y * h])

    # MAR calculation (standard formula using landmarks 13, 14, 78, 308)
    top, bottom = pt(13), pt(14)
    left, right = pt(78), pt(308)
    vertical = np.linalg.norm(top - bottom)
    horizontal = np.linalg.norm(left - right)
    mar = vertical / (horizontal + 1e-6)

    # Crop logic
    mouth_pts = np.array([pt(i) for i in MOUTH_ALL])
    x0, y0 = mouth_pts.min(axis=0).astype(int)
    x1, y1 = mouth_pts.max(axis=0).astype(int)
    pad = 10
    y0c, y1c = max(0, y0 - pad), min(h, y1 + pad)
    x0c, x1c = max(0, x0 - pad), min(w, x1 + pad)
    crop = frame[y0c:y1c, x0c:x1c]

    return mar, crop

URLError: <urlopen error [Errno 11002] getaddrinfo failed>

## 5. Compute the MAR time series for every video

This is the slow step (processes every Nth frame across all videos). Kaggle notebook sessions
have execution time limits (currently 9-12 hrs depending on tier/accelerator) -- for the full
YawDD corpus (~287K frames) budget accordingly, or process a subset first via
`videos = videos[:N]` to test the pipeline end-to-end before committing to a full run.

Adjust `FRAME_STRIDE` to trade off speed vs. temporal resolution -- stride=1 gives the most
accurate yawn-duration measurement, higher strides run faster but can miss short yawns.

In [4]:
from tqdm.notebook import tqdm
import pandas as pd
import os

FRAME_STRIDE = 1
SAVE_CROPS = True

crops_dir = os.path.join(OUTPUT_ROOT, 'mouth_crops')
os.makedirs(crops_dir, exist_ok=True)

# The model was downloaded as 'face_landmarker.task'
MODEL_FILE = 'face_landmarker.task'

if not os.path.exists(MODEL_FILE):
    print(f"Error: {MODEL_FILE} not found. Downloading now...")
    import urllib.request
    model_url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    urllib.request.urlretrieve(model_url, MODEL_FILE)

# Initialize MediaPipe Face Landmarker Task
base_options = python.BaseOptions(model_asset_path=MODEL_FILE)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True,
    num_faces=5)

all_rows = []

with vision.FaceLandmarker.create_from_options(options) as landmarker:
    # Process a subset first for testing (remove [:10] for full run)
    for video_path in tqdm(videos[:320], desc='Videos(subset)'):
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        filename_state = parse_filename_state(video_path)

        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        frame_idx = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % FRAME_STRIDE == 0:
                mar, crop = get_mouth_mar_and_crop(frame, landmarker)
                crop_path = None
                if mar is not None and SAVE_CROPS and crop is not None and crop.size > 0:
                    crop_path = os.path.join(crops_dir, f'{video_name}_f{frame_idx:06d}.jpg')
                    cv2.imwrite(crop_path, crop)
                all_rows.append({
                    'video': video_name,
                    'video_path': video_path,
                    'frame_idx': frame_idx,
                    'fps': fps,
                    'mar': mar,
                    'crop_path': crop_path,
                    'filename_state': filename_state,
                })
            frame_idx += 1
        cap.release()

df = pd.DataFrame(all_rows)
df.to_csv(os.path.join(OUTPUT_ROOT, 'raw_mar_series.csv'), index=False)
print(f'Processed {len(df)} frames across {df["video"].nunique()} videos')
display(df.head())

ModuleNotFoundError: No module named 'pandas'

## 6. Calibrate thresholds on a labeled sample

**Do not skip this.** The thresholds below are placeholders -- they must be tuned against real
examples from your data before the automatic labels mean anything. This cell plots the MAR
distribution split by filename-derived context so you can pick sane starting values, but you
should also visually spot-check crops (Section 8) and adjust.

In [ ]:
import matplotlib.pyplot as plt

# Filter out rows where MAR calculation failed
valid = df.dropna(subset=['mar']).copy()

# Ensure filename_state is treated as a string to avoid grouping issues with None
valid['filename_state'] = valid['filename_state'].fillna('unknown')

fig, ax = plt.subplots(figsize=(10, 5))
groups = valid.groupby('filename_state')

for state, grp in groups:
    ax.hist(grp['mar'], bins=50, alpha=0.5, label=str(state), density=True)

ax.set_xlabel('Mouth Aspect Ratio (MAR)')
ax.set_ylabel('Density')

# Only show legend if we actually have distinct labels
if valid['filename_state'].nunique() > 0:
    ax.legend()

ax.set_title('MAR distribution by filename-derived video state')
plt.savefig(os.path.join(OUTPUT_ROOT, 'mar_distribution.png'), dpi=120, bbox_inches='tight')
plt.show()

# Robust descriptive statistics
stats = groups['mar'].describe()
display(stats)

## 7. Apply the temporal labeling rule

Classifies each frame using:
- **Open/close** via `OPEN_THRESH` on MAR
- **Yawning** = a *sustained* open-mouth stretch (>= `YAWN_MIN_DURATION_SEC` seconds) whose peak
  MAR exceeds `YAWN_MAR_THRESH`
- **Talking** = brief mouth openings/closings with elevated local variance, not sustained wide
  opening
- **mouth_closed** = low, stable MAR with no sustained opening nearby

Tune the constants below after inspecting the histogram above.

In [ ]:
# ---- TUNE THESE after inspecting Section 6's histogram and Section 8's crop review ----
OPEN_THRESH = 0.5                 # MAR above this = mouth "open" for this frame
YAWN_MAR_THRESH = 0.6             # peak MAR required within an open-stretch to call it a yawn
YAWN_MIN_DURATION_SEC = 1.0       # minimum sustained-open duration to call it a yawn (seconds)
TALK_VARIANCE_WINDOW = 5          # +/- frames used to compute local MAR variance
TALK_VARIANCE_THRESH = 0.01       # local variance above this (while mostly closed) => talking

def label_video(video_df, fps):
    video_df = video_df.sort_values('frame_idx').reset_index(drop=True)
    mar = video_df['mar'].values
    n = len(mar)
    labels = np.full(n, 'mouth_closed', dtype=object)
    confidences = np.zeros(n)

    no_face_mask = np.isnan(mar)
    if no_face_mask.all():
        return pd.Series(['no_face']*n), pd.Series([0.0]*n)
    mar_filled = pd.Series(mar).interpolate(limit_direction='both').values

    min_yawn_frames = max(1, int(YAWN_MIN_DURATION_SEC * fps / FRAME_STRIDE))

    is_open = mar_filled > OPEN_THRESH
    i = 0
    while i < n:
        if is_open[i]:
            j = i
            while j < n and is_open[j]:
                j += 1
            duration = j - i
            peak = mar_filled[i:j].max()
            if duration >= min_yawn_frames and peak >= YAWN_MAR_THRESH:
                labels[i:j] = 'yawning'
                confidences[i:j] = np.clip((peak - YAWN_MAR_THRESH) / (1e-6 + peak), 0, 1)
            else:
                labels[i:j] = 'talking'
                confidences[i:j] = 0.6
            i = j
        else:
            w = TALK_VARIANCE_WINDOW
            local = mar_filled[max(0,i-w):i+w+1]
            var = np.var(local)
            if var > TALK_VARIANCE_THRESH:
                labels[i] = 'talking'
                confidences[i] = min(1.0, var / (2*TALK_VARIANCE_THRESH))
            else:
                labels[i] = 'mouth_closed'
                confidences[i] = 1.0 - min(1.0, var / TALK_VARIANCE_THRESH)
            i += 1

    labels[no_face_mask] = 'no_face'
    confidences[no_face_mask] = 0.0
    return pd.Series(labels), pd.Series(confidences)

labeled_frames = []
for video_name, grp in tqdm(df.groupby('video'), desc='Labeling videos'):
    fps = grp['fps'].iloc[0]
    lab, conf = label_video(grp, fps)
    grp = grp.sort_values('frame_idx').reset_index(drop=True)
    grp['label'] = lab.values
    grp['confidence'] = conf.values
    labeled_frames.append(grp)

labeled_df = pd.concat(labeled_frames, ignore_index=True)
labeled_df.to_csv(os.path.join(OUTPUT_ROOT, 'labeled_frames.csv'), index=False)
print(labeled_df['label'].value_counts())

## 8. Flag low-confidence frames for manual review

Anything below `REVIEW_CONFIDENCE_THRESH` gets written to a separate CSV, so a human can correct
the automatic label before you trust the dataset. This mirrors the YawDD+ paper's
human-in-the-loop step, without which their own accuracy numbers don't hold.

**Note on Kaggle:** interactive `ipywidgets` review works in an interactive Kaggle session, but
**not** in a batch "Save & Run All (Commit)" run, since that executes headlessly with no UI. If
you're committing the notebook end-to-end, either run Section 8a manually first in an interactive
session and save `reviewed_labels.csv` back as an input, or skip 8a and treat
`needs_manual_review.csv` as a deliverable to review outside the notebook (e.g. locally, or via a
separate labeling tool/Kaggle dataset export).

In [ ]:
REVIEW_CONFIDENCE_THRESH = 0.7

review_df = labeled_df[
    (labeled_df['confidence'] < REVIEW_CONFIDENCE_THRESH) &
    (labeled_df['label'] != 'no_face')
].copy()

review_df.to_csv(os.path.join(OUTPUT_ROOT, 'needs_manual_review.csv'), index=False)
print(f'{len(review_df)} / {len(labeled_df)} frames ({len(review_df)/len(labeled_df):.1%}) flagged for manual review')
review_df[['video','frame_idx','mar','label','confidence','crop_path']].head(20)

### 8a. Simple in-notebook review widget (interactive sessions only)

Shows a batch of flagged crops with dropdowns to confirm/correct the label. Corrections are
written back into `labeled_df` in the next cell. Re-run this cell with a different
`BATCH_START` to keep reviewing in batches.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

BATCH_START = 0
BATCH_SIZE = 20

review_records = review_df.reset_index(drop=True)
corrections = {}

def show_batch(start):
    clear_output(wait=True)
    end = min(start+BATCH_SIZE, len(review_records))
    if start >= len(review_records):
        print('No more frames to review.')
        return
    for idx in range(start, end):
        row = review_records.loc[idx]
        box_items = []
        if row['crop_path'] and os.path.exists(row['crop_path']):
            img = widgets.Image(value=open(row['crop_path'],'rb').read(), format='jpg', width=120)
            box_items.append(img)
        label_dropdown = widgets.Dropdown(
            options=['mouth_closed','talking','yawning','no_face'],
            value=row['label'] if row['label'] in ['mouth_closed','talking','yawning'] else 'mouth_closed',
            description=f"{row['video']} f{row['frame_idx']}"
        )
        def on_change(change, key=(row['video'], row['frame_idx'])):
            if change['name'] == 'value':
                corrections[key] = change['new']
        label_dropdown.observe(on_change, names='value')
        box_items.append(label_dropdown)
        display(widgets.HBox(box_items))
    print(f'Showing {start}-{end} of {len(review_records)}. Change BATCH_START and re-run to continue.')

show_batch(BATCH_START)

In [ ]:
# Apply corrections made in the widget above back into labeled_df
for (video, frame_idx), new_label in corrections.items():
    mask = (labeled_df['video'] == video) & (labeled_df['frame_idx'] == frame_idx)
    labeled_df.loc[mask, 'label'] = new_label
    labeled_df.loc[mask, 'confidence'] = 1.0  # human-confirmed

labeled_df.to_csv(os.path.join(OUTPUT_ROOT, 'labeled_frames_reviewed.csv'), index=False)
print(f'{len(corrections)} frames corrected by human review.')
print(labeled_df['label'].value_counts())

## 9. Build the final class-folder dataset

Copies each frame's mouth crop (or re-extracts the full frame) into `dataset/{label}/...` so
it's ready for standard `ImageFolder`-style training. If you only need the mouth crops (smaller,
faster to train on), keep `USE_MOUTH_CROP_ONLY = True`.

In [ ]:
import shutil

USE_MOUTH_CROP_ONLY = False   # True = save mouth crop only; False = re-extract & save full frame

final_df = labeled_df[labeled_df['label'].isin(['mouth_closed','talking','yawning'])].copy()

dataset_root = os.path.join(OUTPUT_ROOT, 'dataset')
for cls in ['mouth_closed', 'talking', 'yawning']:
    os.makedirs(os.path.join(dataset_root, cls), exist_ok=True)

if USE_MOUTH_CROP_ONLY:
    for _, row in tqdm(final_df.iterrows(), total=len(final_df), desc='Copying crops'):
        if row['crop_path'] and os.path.exists(row['crop_path']):
            dest = os.path.join(dataset_root, row['label'], os.path.basename(row['crop_path']))
            shutil.copy(row['crop_path'], dest)
else:
    for video_name, grp in tqdm(final_df.groupby('video'), desc='Extracting full frames'):
        video_path = grp['video_path'].iloc[0]
        wanted = set(grp['frame_idx'].tolist())
        cap = cv2.VideoCapture(video_path)
        idx = 0
        wanted_rows = grp.set_index('frame_idx')
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if idx in wanted:
                label = wanted_rows.loc[idx, 'label']
                out_path = os.path.join(dataset_root, label, f'{video_name}_f{idx:06d}.jpg')
                cv2.imwrite(out_path, frame)
            idx += 1
        cap.release()

final_df.to_csv(os.path.join(OUTPUT_ROOT, 'final_labels.csv'), index=False)
print('Final class distribution:')
print(final_df['label'].value_counts())
print(f'Dataset written to: {dataset_root}')

## 10. Package the finished dataset as Kaggle output

Everything under `/kaggle/working/` is automatically preserved when you **Save Version** /
**Commit** the notebook, and appears in the notebook's Output/Data tab -- from there you can
publish it directly as a new Kaggle Dataset for reuse in other notebooks. This cell just zips it
for a single convenient download too.

In [ ]:
zip_path = '/kaggle/working/yawdd_3class_dataset'
shutil.make_archive(zip_path, 'zip', dataset_root)
print(f'Zipped dataset: {zip_path}.zip')
print('Everything in /kaggle/working/ will be saved when you commit this notebook.')
print('You can also publish OUTPUT_ROOT as a new Kaggle Dataset from the Output tab after committing.')

## Accuracy checklist before you trust this dataset

- [ ] Calibrated `OPEN_THRESH`, `YAWN_MAR_THRESH`, `YAWN_MIN_DURATION_SEC` against the MAR
      histogram in Section 6, not left at defaults
- [ ] Manually reviewed **all** flagged low-confidence frames in Section 8 (not just a sample) --
      remember this needs an interactive Kaggle session, not a headless commit run
- [ ] Spot-checked a random sample of *high*-confidence frames too -- automatic confidence
      doesn't guarantee correctness, especially for subjects wearing sunglasses (mouth-only MAR
      is unaffected by glasses, but occasional occlusion from hands/mic can still fool it)
- [ ] Checked class balance (Section 9 output) -- yawning will be a small minority; consider
      stratified splits or class weighting downstream, not naive random shuffling
- [ ] If you obtain the actual YawDD+ release file, cross-validated a subsample of this
      notebook's yawn/no-yawn calls against it
- [ ] If running as a full "Save & Run All" commit, confirmed the run finishes within Kaggle's
      session time limit for your accelerator/tier -- test on a subset of `videos` first